# MiniLM Ranker Training on Google Colab

This trains MiniLM to choose the selected object for each task prompt. Upload the whole `minilm-new-pipeline` folder to Google Drive at:

`MyDrive/minilm-new-pipeline`

Expected files:

- `train_minilm_ranker.py`
- `derived/corrected_coco_tasks_train.jsonl`
- `derived/corrected_coco_tasks_test.jsonl`

The saved checkpoint remains compatible with `inference.py`: `checkpoints/finetuned_text/minilm_task_text_best.pt`.

## 1. Select GPU Runtime

In Colab, use `Runtime > Change runtime type > Hardware accelerator > GPU`, then run the cells below.

In [1]:
!nvidia-smi

Wed May 13 05:25:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.flush_and_unmount()

## 2. Mount Google Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive', force_remount= True)

Mounted at /content/drive


## 3. Configure Project Path

In [4]:
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/minilm-new-pipeline')
assert PROJECT_DIR.exists(), PROJECT_DIR

%cd /content/drive/MyDrive/minilm-new-pipeline
!pwd
!ls -lah train_minilm_ranker.py

/content/drive/MyDrive/minilm-new-pipeline
/content/drive/MyDrive/minilm-new-pipeline
-rw------- 1 root root 15K May 13 04:40 train_minilm_ranker.py


## 4. Install Dependencies

In [5]:
!pip install -q -r requirements-colab.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 30.0 MB/s eta 0:00:00


## 5. Verify Dataset Files

In [6]:
train_jsonl = PROJECT_DIR / 'derived' / 'corrected_coco_tasks_train.jsonl'
val_jsonl = PROJECT_DIR / 'derived' / 'corrected_coco_tasks_test.jsonl'

for path in [train_jsonl, val_jsonl]:
    assert path.exists(), f'Missing file: {path}'
    print(path, 'size:', path.stat().st_size, 'bytes')

/content/drive/MyDrive/minilm-new-pipeline/derived/corrected_coco_tasks_train.jsonl size: 84482291 bytes
/content/drive/MyDrive/minilm-new-pipeline/derived/corrected_coco_tasks_test.jsonl size: 21091487 bytes


## 6. Smoke Test

This checks GPU, paths, model download, data loading, and checkpoint writing before the long run.

In [7]:
import os

os.environ['TRAIN_JSONL'] = str(train_jsonl)
os.environ['VAL_JSONL'] = str(val_jsonl)
os.environ['SAVE_DIR'] = str(PROJECT_DIR / 'checkpoints' / 'finetuned_text_smoke')
os.environ['BATCH_SIZE'] = '8'
os.environ['GRAD_ACCUM_STEPS'] = '2'
os.environ['NUM_EPOCHS'] = '1'
os.environ['MAX_TRAIN_GROUPS'] = '64'
os.environ['MAX_VAL_GROUPS'] = '64'
os.environ['LR'] = '2e-5'
os.environ['RANK_LOSS_WEIGHT'] = '1.0'
os.environ['USE_AMP'] = '1'

!python train_minilm_ranker.py

Device: cuda
GPU: Tesla T4
CUDA runtime: 12.8
Train JSONL: /content/drive/MyDrive/minilm-new-pipeline/derived/corrected_coco_tasks_train.jsonl
Val JSONL: /content/drive/MyDrive/minilm-new-pipeline/derived/corrected_coco_tasks_test.jsonl
Save dir: /content/drive/MyDrive/minilm-new-pipeline/checkpoints/finetuned_text_smoke
Epochs: 1
Batch size in groups: 8
Effective group batch size: 16
Loaded 50305 task/object groups from /content/drive/MyDrive/minilm-new-pipeline/derived/corrected_coco_tasks_train.jsonl
Loaded 12588 task/object groups from /content/drive/MyDrive/minilm-new-pipeline/derived/corrected_coco_tasks_test.jsonl
Using train group limit: 64
Using val group limit: 64
Candidate classes: 362
Positive candidate classes: 20
Negative candidate classes: 342
pos_weight: 17.100000381469727
config.json: 100% 612/612 [00:00<00:00, 3.37MB/s]
tokenizer_config.json: 100% 350/350 [00:00<00:00, 2.22MB/s]
vocab.txt: 232kB [00:00, 26.7MB/s]
tokenizer.json: 466kB [00:00, 61.6MB/s]
special_tokens_

## 7. Full Training

This uses all task prompts and their selected objects. Start with 20 epochs, then continue longer only if validation top-1 accuracy is still improving.

In [8]:
import os

os.environ['TRAIN_JSONL'] = str(train_jsonl)
os.environ['VAL_JSONL'] = str(val_jsonl)
os.environ['SAVE_DIR'] = str(PROJECT_DIR / 'checkpoints' / 'finetuned_text')
os.environ['BATCH_SIZE'] = '16'
os.environ['GRAD_ACCUM_STEPS'] = '4'
os.environ['NUM_EPOCHS'] = '20'
os.environ['MAX_TRAIN_GROUPS'] = '0'
os.environ['MAX_VAL_GROUPS'] = '0'
os.environ['LR'] = '2e-5'
os.environ['RANK_LOSS_WEIGHT'] = '1.0'
os.environ['USE_AMP'] = '1'

!python train_minilm_ranker.py

Device: cuda
GPU: Tesla T4
CUDA runtime: 12.8
Train JSONL: /content/drive/MyDrive/minilm-new-pipeline/derived/corrected_coco_tasks_train.jsonl
Val JSONL: /content/drive/MyDrive/minilm-new-pipeline/derived/corrected_coco_tasks_test.jsonl
Save dir: /content/drive/MyDrive/minilm-new-pipeline/checkpoints/finetuned_text
Epochs: 20
Batch size in groups: 16
Effective group batch size: 64
Loaded 50305 task/object groups from /content/drive/MyDrive/minilm-new-pipeline/derived/corrected_coco_tasks_train.jsonl
Loaded 12588 task/object groups from /content/drive/MyDrive/minilm-new-pipeline/derived/corrected_coco_tasks_test.jsonl
Candidate classes: 202941
Positive candidate classes: 26760
Negative candidate classes: 176181
pos_weight: 6.583744525909424
Loading weights: 100% 103/103 [00:00<00:00, 1992.87it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-

In [9]:
!ls -lh checkpoints/finetuned_text

total 532M
-rw------- 1 root root 266M May 13 08:01 minilm_task_text_best.pt
-rw------- 1 root root 266M May 13 08:08 minilm_task_text_last.pt
